#### Project introduction
**Objective:** To explore the data related to the Titanic disaster: to investigate the charestiristics of passengers and patterns associated with survival rates


**Data source:** [Kaggle](https://www.kaggle.com/)

**Tools & Libraries:** 
1. pandas
2. matplotlib
3. jupyter
4. numpy



#### Data loading

In [209]:
import pandas as pd
import numpy as np

#### Dataset overview

In [210]:
main_df = pd.read_csv("../data/raw/train.csv")
overview_df = pd.DataFrame()
overview_df["Feature"] = main_df.columns
overview_df["Description"] = ["Unique passenger identifier",
                              "Survival status",
                              "Class of ticket",
                              "Passenger name",
                              "Passenger sex",
                              "Passenger age",
                              "Number of siblings/spouses aboard",
                              "Number of parents/children aboard",
                              "Ticket code",
                              "Passenger ticket fare",
                              "Passenger cabin number",
                              "Port of embarkation"
                              ]
overview_df["Data type"] = main_df.dtypes.to_numpy()

print(overview_df)
print(main_df.shape)



        Feature                        Description Data type
0   PassengerId        Unique passenger identifier     int64
1      Survived                    Survival status     int64
2        Pclass                    Class of ticket     int64
3          Name                     Passenger name       str
4           Sex                      Passenger sex       str
5           Age                      Passenger age   float64
6         SibSp  Number of siblings/spouses aboard     int64
7         Parch  Number of parents/children aboard     int64
8        Ticket                        Ticket code       str
9          Fare              Passenger ticket fare   float64
10        Cabin             Passenger cabin number       str
11     Embarked                Port of embarkation       str
(891, 12)


#### Data Quality Assessment

In [211]:
verif_df = pd.DataFrame()
verif_df["Missing values"] = main_df.isna().sum()
verif_df["% Missing values"] = round(verif_df["Missing values"] / main_df.shape[0] * 100, 2)
rules = {
    "PassengerId":lambda x: x > 0,
    "Survived":lambda x: x in [0, 1],
    "Pclass":lambda x: x in [1, 2, 3],
    "Name":lambda x: type(x) is str,
    "Sex":lambda x: x in ["male", "female"],
    "Age":lambda x: x >= 0,
    "SibSp":lambda x: x >= 0,
    "Parch":lambda x: x >= 0,
    "Ticket":lambda x: type(x) is str,
    "Fare":lambda x: x >= 0,
    "Cabin":lambda x: type(x) is str,
    "Embarked":lambda x: x in ["S", "Q", "C"],
}
verif_df["Completeness"] = pd.Series({
    col: main_df[col].dropna().apply(rule).all()
    for col, rule in rules.items()
})
print(verif_df)

print(main_df.isna().sum().sum() / main_df.size * 100)
print(main_df.duplicated().sum())
print(main_df["PassengerId"].duplicated().sum())
print(main_df.isin(["?", "-", "n/a", "unknown", "None", "null", ""]).sum().sum())

             Missing values  % Missing values  Completeness
PassengerId               0              0.00          True
Survived                  0              0.00          True
Pclass                    0              0.00          True
Name                      0              0.00          True
Sex                       0              0.00          True
Age                     177             19.87          True
SibSp                     0              0.00          True
Parch                     0              0.00          True
Ticket                    0              0.00          True
Fare                      0              0.00          True
Cabin                   687             77.10          True
Embarked                  2              0.22          True
8.099513655069211
0
0
0


#### Descritive Statistics

In [212]:
int_stats_df = pd.DataFrame()
int_stats_df["Ср. значения"] = main_df.drop(columns=["PassengerId"]).mean(numeric_only=True)
int_stats_df["Станд. откл."] = main_df.std(numeric_only=True)
int_stats_df["Минимум"] = main_df.drop(columns=["Pclass", "Survived"]).min(numeric_only=True)
int_stats_df["Максимум"] = main_df.drop(columns=["Pclass", "Survived"]).max(numeric_only=True)
int_stats_df["Медиана"] = main_df.drop(columns=["Pclass", "Survived"]).median(numeric_only=True)
#Я думаю, что нужно оставить именно эти показатели, я могу вывести минимум и максимум для статуса выживания или класса билета, но это ничего не даст, поэтому я намеренно даю nan

str_stats_df = pd.DataFrame({
    "Number of people" : {
        "Total": 891,
        "1 class": (main_df["Pclass"] == 1).sum(),
        "2 class": (main_df["Pclass"] == 2).sum(),
        "3 class": (main_df["Pclass"] == 3).sum(),
        "Adult men": ((main_df["Sex"] == "male") & (main_df["Age"] >= 18)).sum(),
        "Adult women": ((main_df["Sex"] == "female") & (main_df["Age"] >= 18)).sum(),
        "Children": (main_df["Age"] < 18).sum(),
        "Unknown age": (main_df["Age"].isna()).sum(),
        '"S" port': (main_df["Embarked"] == "S").sum(),
        '"C" port': (main_df["Embarked"] == "C").sum(),
        '"Q" port': (main_df["Embarked"] == "Q").sum()
    }
})

print(int_stats_df)
print(str_stats_df)

          Ср. значения  Станд. откл.  Минимум  Максимум  Медиана
Survived      0.383838      0.486592      NaN       NaN      NaN
Pclass        2.308642      0.836071      NaN       NaN      NaN
Age          29.699118     14.526497     0.42   80.0000  28.0000
SibSp         0.523008      1.102743     0.00    8.0000   0.0000
Parch         0.381594      0.806057     0.00    6.0000   0.0000
Fare         32.204208     49.693429     0.00  512.3292  14.4542
             Number of people
Total                     891
1 class                   216
2 class                   184
3 class                   491
Adult men                 395
Adult women               206
Children                  113
Unknown age               177
"S" port                  644
"C" port                  168
"Q" port                   77


#### Survival Distribution

In [217]:
mask = main_df["Survived"]
survived_stats_df = pd.DataFrame({
    "Number of survivors" : {
        "Total": (mask).sum(),
        "1 class": ((mask) & (main_df["Pclass"] == 1)).sum(),
        "2 class": ((mask) & (main_df["Pclass"] == 2)).sum(),
        "3 class": ((mask) & (main_df["Pclass"] == 3)).sum(),
        "Adult men": ((mask) & (main_df["Sex"] == "male") & (main_df["Age"] >= 18)).sum(),
        "Adult women": ((mask) & (main_df["Sex"] == "female") & (main_df["Age"] >= 18)).sum(),
        "Children": ((mask) & (main_df["Age"] < 18)).sum(),
        "Unknown age": ((mask) & (main_df["Age"].isna())).sum(),
        '"S" port': ((mask) & (main_df["Embarked"] == "S")).sum(),
        '"C" port': ((mask) & (main_df["Embarked"] == "C")).sum(),
        '"Q" port': ((mask) & (main_df["Embarked"] == "Q")).sum()
    }
})
survived_stats_df["% survivors"] = survived_stats_df["Number of survivors"] / str_stats_df["Number of people"] * 100
survived_stats_df["Number of victims"] = str_stats_df["Number of people"] - survived_stats_df["Number of survivors"]
survived_stats_df["% victims"] = survived_stats_df["Number of victims"] / str_stats_df["Number of people"] * 100

print(survived_stats_df)


             Number of survivors  % survivors  Number of victims  % victims
Total                        342    38.383838                549  61.616162
1 class                      136    62.962963                 80  37.037037
2 class                       87    47.282609                 97  52.717391
3 class                      119    24.236253                372  75.763747
Adult men                     70    17.721519                325  82.278481
Adult women                  159    77.184466                 47  22.815534
Children                      61    53.982301                 52  46.017699
Unknown age                   52    29.378531                125  70.621469
"S" port                     217    33.695652                427  66.304348
"C" port                      93    55.357143                 75  44.642857
"Q" port                      30    38.961039                 47  61.038961


#### Initial Findings 
The libraries specified in the Data Loading section were used to work with the dataset

The dataset is a table that consists of 891 rows describing passengers and 12 columns describing different characteristics of these passengers. A description of these 12 columns was prepared

The assessment of data completeness showed that most columns were well-filled. 866 missing values were identified: Age - 177(19.87%), Cabin - 687(77.10%), Embarked - 2(0.22%). Missing values account for 8.1% of the total number of values. No duplicate rows or IDs were found. No invalid values were found either("?", "-", "n/a", "unknown", "None", "null", "").

The initial statistics on the number of people in different categories were collected. Mean values, standard deviations, minimum, maximum and median for columns were calculated

The survival statistics for different groups(age, ticket class, sex etc.) were calculated